In [1]:
# ---- Cell 1: Install MCP + agents packages ----
%pip install mcp azure-ai-agents azure-identity --quiet
print("✅ Done")

Note: you may need to restart the kernel to use updated packages.
✅ Done


In [2]:
# This installs all four Azure packages you need for the entire course

%pip install python-dotenv
%pip install azure-ai-inference
%pip install azure-identity  
%pip install azure-ai-projects>= 2.0.0
%pip install openai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement 2.0.0 (from versions: none)
ERROR: No matching distribution found for 2.0.0
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
# ---- Cell 2: Imports ----

import json                          # convert Python dicts to JSON strings
import asyncio                       # asyncio runs async code — MCP uses async throughout
import os                            # read environment variables
from dotenv import load_dotenv       # load .env file

load_dotenv()                        # load PROJECT_ENDPOINT and MODEL_DEPLOYMENT_NAME

PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
MODEL = os.environ["MODEL_DEPLOYMENT_NAME"]

print("✅ Imports done")
print("Endpoint:", PROJECT_ENDPOINT)

✅ Imports done
Endpoint: https://abhishekbhalotia333-344-resource.services.ai.azure.com/api/projects/abhishekbhalotia333-3440


In [2]:
# ---- Cell 3: Define your tool functions (same logic as Module 2) ----
# Nothing new here — same get_mbds and calculate_cvs from last notebook
# The difference comes in HOW we expose them (MCP server vs manual schema)

import random

def get_mbds(region: str) -> str:
    """
    Retrieves active MBDs for a given region from the D2O database.
    Use when the user asks about MBDs in a specific region.
    Args:
        region: Region code — APAC, EMEA, NA, LATAM, or GLOBAL
    Returns:
        JSON string with list of active MBDs for that region.
    """
    fake_db = {
        "APAC": [
            {"id": "MBD-APAC-0042", "name": "[APAC] Premium Electronics",  "category": "CAT-ELEC"},
            {"id": "MBD-APAC-0087", "name": "[APAC] Food & Beverage Core", "category": "CAT-FOOD"},
            {"id": "MBD-APAC-0103", "name": "[APAC] Health & Wellness",    "category": "CAT-HLTH"},
        ],
        "EMEA": [
            {"id": "MBD-EMEA-0011", "name": "[EMEA] Apparel Premium",      "category": "CAT-APPR"},
            {"id": "MBD-EMEA-0034", "name": "[EMEA] Consumer Electronics", "category": "CAT-ELEC"},
        ],
        "NA": [
            {"id": "MBD-NA-0021",   "name": "[NA] Automotive Parts",       "category": "CAT-AUTO"},
            {"id": "MBD-NA-0055",   "name": "[NA] Home & Living Core",     "category": "CAT-HOME"},
        ],
    }
    result = fake_db.get(region.upper(), [])
    if not result:
        return json.dumps({"error": f"No MBDs found for region: {region}"})
    return json.dumps({"region": region, "mbds": result, "count": len(result)})


def calculate_cvs(mbd_ids: str) -> str:
    """
    Calculates CV (Coefficient of Variation) for a list of MBDs.
    Use when the user wants CV values, stability checks, or tier classification.
    Args:
        mbd_ids: Comma-separated MBD IDs e.g. "MBD-APAC-0042,MBD-APAC-0087"
    Returns:
        JSON string with CV%, tier, and recommended action per MBD.
    """
    ids = [mid.strip() for mid in mbd_ids.split(",")]
    results = []
    for mbd_id in ids:
        if "APAC" in mbd_id:
            mean, std_dev = 4520000, random.uniform(400000, 900000)
        elif "EMEA" in mbd_id:
            mean, std_dev = 7800000, random.uniform(600000, 1800000)
        else:
            mean, std_dev = 3200000, random.uniform(300000, 800000)
        cv = (std_dev / mean) * 100
        if cv <= 10:
            tier, action = "Tier 1 — Stable",     "None required"
        elif cv <= 20:
            tier, action = "Tier 2 — Acceptable",  "Monitor in next review cycle"
        elif cv <= 35:
            tier, action = "Tier 3 — Elevated",    "Investigate root cause within 30 days"
        else:
            tier, action = "Tier 4 — Critical",    "Immediate data quality review required"
        results.append({
            "mbd_id": mbd_id,
            "cv_percent": round(cv, 2),
            "tier": tier,
            "action": action
        })
    return json.dumps({"cv_results": results})

print("✅ Functions defined")

✅ Functions defined


In [4]:
# ---- Cell 3b: Fix async in Jupyter ----
%pip install nest_asyncio --quiet

import nest_asyncio
nest_asyncio.apply()
# nest_asyncio patches Jupyter's event loop so asyncio.run() works inside notebooks
# Without this, asyncio.run() crashes because Jupyter already owns the event loop

print("✅ nest_asyncio applied")

Note: you may need to restart the kernel to use updated packages.
✅ nest_asyncio applied


In [5]:
# ---- Cell 4: Build the MCP Server ----
# This is what's NEW in Module 3 vs Module 2.
# In Module 2: you manually wrote the tool schema (the "tools" list) 
#              and manually ran the function when agent said "requires_action"
# In Module 3: MCP Server does BOTH automatically
#   - it reads your function's docstring → generates the schema for you
#   - it runs the function when the agent calls it → no requires_action loop needed

from mcp.server.fastmcp import FastMCP  
# FastMCP is a lightweight MCP server library
# "fast" = minimal setup — decorate a function, it becomes an MCP tool instantly

# Create the MCP server — give it a name (shows up in logs and agent discovery)
mcp_server = FastMCP("d2o-tools-server")
# Think of mcp_server as a waiter who knows your entire menu (your tools)
# Any agent that connects to this server can see and call all tools on the menu

# The @mcp_server.tool() decorator is the magic line
# It does two things:
#   1. Registers this function as an MCP tool (adds it to the server's menu)
#   2. Reads the docstring automatically to generate the tool description
#      — you don't write a separate schema like in Module 2

@mcp_server.tool()
def get_mbds_tool(region: str) -> str:
    # This wrapper just calls our actual get_mbds function
    # We use a wrapper so the MCP tool name is "get_mbds_tool" 
    # and our core function stays clean and reusable
    """
    Retrieves active MBDs for a given region from the D2O database.
    Use when the user asks about MBDs in a specific region.
    Args:
        region: Region code — APAC, EMEA, NA, LATAM, or GLOBAL
    """
    return get_mbds(region)   # delegates to the function we defined in Cell 3

@mcp_server.tool()
def calculate_cvs_tool(mbd_ids: str) -> str:
    """
    Calculates CV for a list of MBDs. Use when CV values or tier 
    classification is requested.
    Args:
        mbd_ids: Comma-separated MBD IDs e.g. "MBD-APAC-0042,MBD-APAC-0087"
    """
    return calculate_cvs(mbd_ids)   # delegates to function from Cell 3

print("✅ MCP server defined with tools:")

# List all tools registered on the server — confirms both are visible
import asyncio

async def show_tools():
    tools = await mcp_server.list_tools()   
    # list_tools() is async — it returns the full tool catalogue
    # async means: don't freeze while waiting, let other things run
    for tool in tools:
        print(f"  • {tool.name}: {tool.description[:60]}...")
        # tool.name = what the agent sees when it discovers this server
        # tool.description = auto-generated from your docstring

asyncio.run(show_tools())
# asyncio.run() is how you run an async function from regular (non-async) code
# In a Jupyter notebook this sometimes needs a workaround — if you get an error
# about "event loop already running", add: import nest_asyncio; nest_asyncio.apply()

✅ MCP server defined with tools:
  • get_mbds_tool: 
    Retrieves active MBDs for a given region from the D2O d...
  • calculate_cvs_tool: 
    Calculates CV for a list of MBDs. Use when CV values or...


In [ ]:
# ---- Cell 5a: Find what MCP-related classes exist in your SDK ----
import azure.ai.agents.models as m

# List everything in the models module that mentions mcp, tool, or function
mcp_related = [x for x in dir(m) if any(k in x.lower() for k in ['mcp', 'tool', 'function'])]
print("Available tool-related classes:")
for item in mcp_related:
    print(f"  {item}")

In [ ]:
# ---- Cell 5: Start the MCP server and get tool definitions ----
# In Module 2 we manually built the "tools" list (30 lines of schema)
# Here the MCP server generates that list automatically from our decorators

from azure.ai.agents import AgentsClient
from azure.ai.agents.models import McpTool   
# McpTool is the Azure SDK class that wraps an MCP server
# It knows how to talk to MCP servers using the standard protocol

from azure.identity import DefaultAzureCredential
from mcp import ClientSession                
# ClientSession = the MCP CLIENT side — connects to our server
# Our server = mcp_server (defined in Cell 4)
# ClientSession talks TO the server, discovers its tools, calls them

from mcp.client.stdio import stdio_client    
# stdio = "standard input/output" — the transport layer
# Our server and client communicate through stdin/stdout pipes
# Think of it as two programs talking to each other through a tube

# Create the Azure agents client — same as Module 2
agents_client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential()
)
print("✅ AgentsClient created")

async def run_mcp_agent():
    # Everything inside here is async — it runs inside the event loop
    
    # Step 1: Start the MCP server as a subprocess
    # mcp_server.run() starts our FastMCP server
    # stdio_client() creates a client that connects to it via stdin/stdout
    # This gives us (read_stream, write_stream) — the two-way communication pipe
    server_params = mcp_server.run()         
    # server_params tells the client HOW to start and talk to the server
    
    async with stdio_client(server_params) as (read, write):
        # read = incoming messages FROM the server
        # write = outgoing messages TO the server
        
        async with ClientSession(read, write) as session:
            # ClientSession manages the MCP conversation protocol
            # initialize() does the handshake — "hello server, what tools do you have?"
            await session.initialize()
            
            # Step 2: Wrap the MCP session as an Azure McpTool
            # McpTool translates between MCP protocol and Azure Agents SDK
            # It fetches the tool list from the server and formats it for the agent
            mcp_tool = McpTool(session)
            await mcp_tool.refresh_tools()   
            # refresh_tools() asks the server: "list all your tools"
            # Result is stored inside mcp_tool — agent will read it automatically
            
            print("✅ MCP session established")
            print(f"Tools discovered: {len(mcp_tool.definitions)} tools")
            # mcp_tool.definitions = the tool schema list, auto-generated
            # This REPLACES the 30-line "tools" list we wrote manually in Module 2
            
            for tool_def in mcp_tool.definitions:
                print(f"  • {tool_def['function']['name']}")
            
            # Step 3: Create the agent — notice tools=mcp_tool instead of tools=tools
            agent = agents_client.create_agent(
                model=MODEL,
                name="ab-mcp-agent",
                instructions=(
                    "You are the D2O Data Assistant. "
                    "Use get_mbds_tool when a region is mentioned. "
                    "Use calculate_cvs_tool when CV calculations are requested. "
                    "Always present tier classification and recommended actions."
                ),
                tools=mcp_tool.definitions   
                # mcp_tool.definitions = same format as the manual "tools" list
                # but auto-generated by the MCP server from our decorators
            )
            print(f"✅ Agent created: {agent.id}")
            
            # Step 4: Create thread and send message — identical to Module 2
            thread = agents_client.threads.create()
            agents_client.messages.create(
                thread_id=thread.id,
                role="user",
                content="Show me all MBDs for EMEA, then calculate their CVs"
                # One message asking for TWO tool calls in sequence
                # Agent will call get_mbds_tool first, then calculate_cvs_tool
            )
            print(f"✅ Thread created: {thread.id}")
            
            # Step 5: Run with MCP tool execution
            # process_mcp_tool_calls=True is the KEY DIFFERENCE from Module 2
            # In Module 2: you wrote the requires_action loop yourself
            # Here: the SDK handles requires_action automatically using the MCP session
            run = agents_client.runs.create_and_process(
                thread_id=thread.id,
                agent_id=agent.id,
                tools=mcp_tool              
                # passing mcp_tool (not mcp_tool.definitions) here
                # so the SDK knows HOW to execute the tools via MCP protocol
            )
            # create_and_process = create the run AND handle all tool calls automatically
            # No polling loop needed — SDK does it all inside this one call
            print(f"✅ Run complete: {run.status}")
            
            # Step 6: Read reply — identical to Module 2
            messages = agents_client.messages.list(thread_id=thread.id)
            for msg in messages:
                if msg.role == "assistant":
                    for block in msg.content:
                        if hasattr(block, "text"):
                            print("\n--- Agent reply ---")
                            print(block.text.value)
                    break
            
            # Step 7: Cleanup
            agents_client.threads.delete(thread.id)
            agents_client.delete_agent(agent.id)
            print("✅ Cleaned up")

print("✅ Cell 5 defined — run Cell 6 to execute")

In [ ]:
# ---- Cell 5: Connect MCP tools to agent using ToolSet + FunctionTool ----
# McpTool not in azure-ai-agents v2.0.1 — use FunctionTool + ToolSet instead
# FunctionTool: wraps Python functions, auto-generates schema from docstrings
# ToolSet: groups multiple tools together for an agent
# The MCP server (Cell 4) still runs — it's the tool host
# ToolSet is how the Azure SDK connects those tools to an agent in v2.0.1

from azure.ai.agents import AgentsClient
from azure.ai.agents.models import FunctionTool, ToolSet
# FunctionTool — takes a SET of Python functions, reads their docstrings,
#                auto-generates the schema (replaces our 30-line manual schema from M2)
# ToolSet — a container that holds multiple tool types together
#            (you could add FileSearchTool + FunctionTool + CodeInterpreterTool in one ToolSet)

from azure.identity import DefaultAzureCredential

agents_client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential()
)

# FunctionTool reads the docstrings from get_mbds_tool and calculate_cvs_tool
# and auto-generates the JSON schema the agent needs
# Notice: we pass a SET of functions — Python set syntax uses {}
functions = FunctionTool(functions={get_mbds_tool, calculate_cvs_tool})
# get_mbds_tool and calculate_cvs_tool are the decorated functions from Cell 4
# FunctionTool introspects them — reads name, docstring, type hints
# and generates the same "tools" list we wrote manually in Module 2

# ToolSet groups tools — in production you'd combine multiple tool types here
toolset = ToolSet()
toolset.add(functions)   # add our FunctionTool to the set

print("✅ ToolSet created")
print(f"Tools in set: {[t.function.name for t in functions.definitions]}")
# functions.definitions = the auto-generated schema list — same format as Module 2

# Create the agent — tools=toolset instead of tools=tools (the manual list)
agent = agents_client.create_agent(
    model=MODEL,
    name="ab-mcp-agent",
    instructions=(
        "You are the D2O Data Assistant. "
        "Use get_mbds_tool when a region is mentioned. "
        "Use calculate_cvs_tool when CV calculations are requested. "
        "Present tier classification and recommended actions."
    ),
    tools=toolset.definitions,   # auto-generated from docstrings — no manual schema
    tool_resources=toolset.resources
)
print(f"✅ Agent created: {agent.id}")

# Thread + message — identical to Module 2
thread = agents_client.threads.create()
agents_client.messages.create(
    thread_id=thread.id,
    role="user",
    content="Show me all EMEA MBDs then calculate their CVs"
)
print(f"✅ Thread created: {thread.id}")

In [ ]:
# ---- Cell 6: Run with automatic tool handling ----
# create_and_process = start run AND automatically handle requires_action
# The SDK polls internally and calls our functions when needed
# This replaces the entire while loop from Module 2

import json, time

run = agents_client.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id,
    toolset=toolset    # SDK uses toolset to know HOW to execute function calls
    # when agent says "requires_action: call get_mbds_tool"
    # the SDK finds get_mbds_tool in toolset and calls it automatically
    # no manual submit_tool_outputs needed
)
print(f"✅ Run complete: {run.status}")

# Read reply
messages = agents_client.messages.list(thread_id=thread.id)
for msg in messages:
    if msg.role == "assistant":
        for block in msg.content:
            if hasattr(block, "text"):
                print("\n--- Agent reply ---")
                print(block.text.value)
        break

New code